In [1]:
import sys
import yaml
import xarray as xr
import numpy as np
import pandas as pd
import itertools
%matplotlib inline
# import wrf

# plot styles/formatting
import seaborn as sns
import cmocean.cm as cmo
import cmocean

# matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar # different way to handle colorbar
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from matplotlib import cm

# cartopy
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
import cartopy.feature as cfeature


# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
import global_vars
from plotter import draw_basemap
import load_composites as lc

In [2]:
path_to_data = global_vars.path_to_data
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write
path_to_figs = '../figs/'      # figures

In [3]:
region = 'gulf_of_mexico'
ARDT = 'tARget'

## load ar dates with region (include HUC8 and start date for adding trajectories)
fname = '../out/bbox_dates_{0}_full_{1}.csv'.format(region, ARDT)
df = pd.read_csv(fname)
df['day'] = pd.to_datetime(df['time']).dt.normalize()

## make a copy of the df but keep only time/index
d = {'datetime': df.day.values}
ar_dates = pd.DataFrame(d)
ar_dates = ar_dates.drop_duplicates(subset=['datetime'])
ar_dates = ar_dates.sort_values(by='datetime')
idx = (ar_dates.datetime.dt.month >= 11) | (ar_dates.datetime.dt.month <= 4)
ar_dates = ar_dates.loc[idx]
ar_dates

,datetime
1201,2007-03-21
240,2007-03-22
1644,2007-03-23
2959,2007-03-27
2957,2007-03-28
1217,2010-03-07
2085,2019-03-10


In [4]:
# import configuration file for case study choice
yaml_doc = '../data/domains.yml'
config = yaml.load(open(yaml_doc), Loader=yaml.SafeLoader)

FileNotFoundError: [Errno 2] No such file or directory: '../data/domains.yml'

In [ ]:
def create_horizontal_composite_plots(ds_hc, ds_hc_anom, ds_tval, ssn, lag, ext):
    # Set up projection
    # mapcrs = ccrs.Mercator()
    mapcrs = ccrs.PlateCarree()
    datacrs = ccrs.PlateCarree()
    
    # Set lat/lons
    lats = ds_hc.latitude.values
    lons = ds_hc.longitude.values
    
    # Set tick/grid locations
    tx = 10
    ty = 5
    dx = np.arange(ext[0],ext[1]+tx,tx)
    dy = np.arange(ext[2],ext[3]+ty,ty)
    
    # list of letters to append to titles
    letter_lst = list(map(chr, range(97, 123)))
    
    # Create figure
    fig = plt.figure(figsize=(7.75, 7.))
    fig.dpi = 300
    fname = path_to_figs + '{0}_IVT_700z_composite_lag{1}'.format(ssn, lag)
    fmt = 'png'
    
    # contour labels
    kw_clabels = {'fontsize': 7, 'inline': True, 'inline_spacing': 7, 'fmt': '%i',
                  'rightside_up': True, 'use_clabeltext': True}
    
    nrows = 5
    ncols = 2
    
    ## Use gridspec to set up a plot with a series of subplots that is
    ## n-rows by n-columns
    gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
    ## use gs[rows index, columns index] to access grids
    
    #############################
    ### NON-ANOMALY COMPOSITE ###
    #############################
    
    rowidx = [0, 1, 2]
    region_lst = ['pnw', 'baja','gulf_of_mexico']
    region_lbl = ['Pacific Northwest', 'Baja', 'Gulf of Mexico']
    blons = [False, False, True]
    lbl_lst = [0, 2, 4]
    for i, region in enumerate(region_lst):
        ds = ds_hc.sel(region=region, lag=lag)
        ax = fig.add_subplot(gs[rowidx[i], 0], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=True, right_lats=False, bottom_lons=blons[i])
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
        
        # Contour Filled
        lats = ds.latitude.values
        lons = ds.longitude.values
        ivt = ds.ivt.values
        cflevs = np.arange(100, 450, 50) # levels for IVT
        cmap = cmo.deep # cmap for IVT
        cf = ax.contourf(lons, lats, ivt, transform=datacrs,
                         levels=cflevs, cmap=cmap, alpha=0.9, extend='max')
    
        # Wind barbs / vectors 
        uvec = ds.ivtu.values
        vvec = ds.ivtv.values
        # uvec_mask = ds.IVTu.where((ds.IVT >=250.)).values # mask values where IVT magnitude is less than 250 kg m-1 s-1
        # vvec_mask = ds.IVTv.where((ds.IVT >=250.)).values # mask values where IVT magnitude is less than 250 kg m-1 s-1
    
        Q = ax.quiver(lons, lats, uvec, vvec, transform=datacrs, 
                  color='k', regrid_shape=20,
                  angles='xy', scale_units='xy', scale=125, units='xy')
    
        # Contour Lines
        hgts = ds.z.values/9.80665 ## convert to geopotential height
        hgts = hgts/(10) # convert to meters # 750-hPa Heights
        # print(hgts.min(), hgts.max())
        clevs = np.arange(0, 1280, 4)
        cs = ax.contour(lons, lats, hgts, transform=datacrs,
                        levels=clevs, colors='grey', linewidths=0.7)
        
        kw_clabels = {'fontsize': 8.5, 'inline': True, 'inline_spacing': 5, 'fmt': '%i', 'rightside_up': True, 'use_clabeltext': True}
        cl = ax.clabel(cs, clevs[::2], **kw_clabels)
        for txt in cl:
                    txt.set_bbox(dict(facecolor='white', edgecolor='none', pad=0.5))
    
        ext3 = config[region]['ext']
        ax.add_patch(mpatches.Rectangle(xy=[ext3[0], ext3[2]], width=ext3[1]-ext3[0], height=ext3[3]-ext3[2],
                                            fill=False,
                                            edgecolor='r',
                                            linewidth=0.75,
                                            transform=datacrs,
                                            zorder=199))
    
        titlestring = '({0}) n={1}'.format(letter_lst[lbl_lst[i]], str(ds.ndays.values))
        ax.text(0.025, 0.96, titlestring, ha='left', va='top', transform=ax.transAxes, fontsize=10., backgroundcolor='white', zorder=101)
            
        ax.text(-0.16, 0.5, region_lbl[i], va='bottom', ha='center',
            rotation='vertical', rotation_mode='anchor', fontsize=11,
            transform=ax.transAxes)
    
    # quiver key
    qk = ax.quiverkey(Q, 0.75, -.15, 250, '250 kg m$^{-1}$ s$^{-1}$', labelpos='E',
                      coordinates='axes', fontproperties={'size': 6.0})
    # Colorbar (single)
    cbax = plt.subplot(gs[-1,0]) # colorbar axis
    cb = fig.colorbar(cf, cbax, orientation='horizontal', drawedges=False)
    cb.set_label('IVT (kg m$^{-1}$ s$^{-1}$)', fontsize=11)
    cb.ax.tick_params(labelsize=12)
    
    #########################
    ### ANOMALY COMPOSITE ###
    #########################
    lbl_lst = [1, 3, 5]
    for i, region in enumerate(region_lst):
        ds = ds_hc_anom.sel(region=region, lag=lag)
        tval = ds_tval.sel(region=region, lag=lag)
        ax = fig.add_subplot(gs[rowidx[i], 1], projection=mapcrs)
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=False, right_lats=False, bottom_lons=blons[i])
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
        
        # Contour Filled
        lats = ds.latitude.values
        lons = ds.longitude.values
        ivt = ds.ivt.values
        # print(ivt.min(), ivt.max())
        cmap = cm.BrBG
        bnds = np.arange(-100., 110., 10)
        cbarticks = bnds[::2]
        norm = mcolors.BoundaryNorm(bnds, cmap.N)
        cf = ax.contourf(lons, lats, ivt, transform=datacrs,
                         levels=bnds, cmap=cmap, norm=norm, alpha=0.9, extend='both')
    
        # Wind barbs / vectors 
        uvec_mask = ds.ivtu.where((tval.ivtu == True) | (tval.ivtv == True)).values
        vvec_mask = ds.ivtv.where((tval.ivtu == True) | (tval.ivtv == True)).values
    
        Q = ax.quiver(lons, lats, uvec_mask, vvec_mask, transform=datacrs, 
                  color='k', regrid_shape=20,
                  angles='xy', scale_units='xy', scale=50, units='xy')
    
        # Contour Lines
        hgts = ds.z.values/9.80665 ## convert to geopotential height
        hgts = hgts/(10) # convert to meters # 750-hPa Heights
        # print(hgts.min(), hgts.max())
        clevs = np.arange(-20, 22, 2)
        cs = ax.contour(lons, lats, hgts, transform=datacrs,
                        levels=clevs, colors='grey', linewidths=0.7)
        
        kw_clabels = {'fontsize': 8.5, 'inline': True, 'inline_spacing': 5, 'fmt': '%i', 'rightside_up': True, 'use_clabeltext': True}
        cl = ax.clabel(cs, clevs[::2], **kw_clabels)
        for txt in cl:
                    txt.set_bbox(dict(facecolor='white', edgecolor='none', pad=0.5))
    
        ext3 = config[region]['ext']
        ax.add_patch(mpatches.Rectangle(xy=[ext3[0], ext3[2]], width=ext3[1]-ext3[0], height=ext3[3]-ext3[2],
                                            fill=False,
                                            edgecolor='r',
                                            linewidth=0.75,
                                            transform=datacrs,
                                            zorder=199))
    
        titlestring = '({0}) n={1}'.format(letter_lst[lbl_lst[i]], str(ds.ndays.values))
        ax.text(0.025, 0.96, titlestring, ha='left', va='top', transform=ax.transAxes, fontsize=10., backgroundcolor='white', zorder=101)
    
    # quiver key
    qk = ax.quiverkey(Q, 0.75, -.15, 50, '50 kg m$^{-1}$ s$^{-1}$', labelpos='E',
                      coordinates='axes', fontproperties={'size': 6.0})
    # Colorbar (single)
    cbax = plt.subplot(gs[-1,1]) # colorbar axis
    cb = fig.colorbar(cf, cbax, orientation='horizontal', drawedges=False)
    cb.set_label('IVT (kg m$^{-1}$ s$^{-1}$)', fontsize=11)
    cb.ax.tick_params(labelsize=12)
    
    fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi)
    
    # Show
    # plt.show()

In [ ]:
ext = [-140., -90., 20, 50]
ds_hc = lc.load_non_anomaly_composites('NDJFMA', ext)
ds_hc.sel(region='gulf_of_mexico')

In [ ]:
##################################
### LOAD HORIZONTAL COMPOSITES ###
##################################
ext = [-140., -90., 20, 50]
ssn_lst = ['NDJFMA', 'MJJASO']
lag_lst = [0, 1]

for i, ssn in enumerate(ssn_lst):
    for j, lag in enumerate(lag_lst):
        ds_hc = lc.load_non_anomaly_composites(ssn, ext)
        ds_hc_anom, ds_tval = lc.load_anomaly_composites(ssn, ext)

        create_horizontal_composite_plots(ds_hc, ds_hc_anom, ds_tval, ssn, lag, ext)